In [1]:
import numpy as np
import pandas as pd
import torch
import pickle
from pathlib import Path

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '4_Baselines' / '4.1_Matrix_Factorization'))
import MF_class as MF

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

# 1. Load Data

In [ ]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Simulation'

data = pd.read_csv(data_path / 'simulation_data.csv')

n_users = data['user_id'].nunique()
n_items = data['item_id'].nunique()
print(f'Number of users: {n_users}, Number of items: {n_items}')

Number of users: 5000, Number of items: 3000


In [3]:
data['interaction'] = 1
data_matrix = data.pivot_table(index='user_id', columns='item_id', values='interaction').fillna(0).astype(int)
melted = data_matrix.melt(ignore_index=False).reset_index()

In [4]:
rng = np.random.default_rng(seed=42)
valid_idx = rng.choice(melted.index, size=int(0.1 * len(melted)), replace=False)
train_idx = melted.index.difference(valid_idx)

train = melted.loc[train_idx].reset_index(drop=True)
valid = melted.loc[valid_idx].reset_index(drop=True)

# 2. Train Model

In [5]:
model = MF.MatrixFactorizationTorch(
    n_users=n_users, 
    n_items=n_items, 
    n_factors=40
)

model.fit(
    train_data=train.values,
    val_data=valid.values,
    lr=5e-3, 
    wd=1e-7,
    pos_weight=1,
    batch_size=2**15,
    n_epochs=50,
    device=torch.device('cuda:0'), 
    use_amp=True)

Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || BCE    | BCE-POS | BCE-NEG | MPR    || BCE    | BCE-POS | BCE-NEG | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
   1   || 0.0578 |  0.6736 |  0.0254 | 0.9301 || 0.0592 |  0.6984 |  0.0255 | 0.9247 || 254.80  | None  | 00:02.45
   2   || 0.0469 |  0.5941 |  0.0180 | 0.9431 || 0.0505 |  0.6574 |  0.0186 | 0.9258 ||  58.51  | 0.658 | 00:04.53
   3   || 0.0452 |  0.5676 |  0.0177 | 0.9509 || 0.0504 |  0.6587 |  0.0184 | 0.9268 ||  30.30  | 0.478 | 00:06.48
   4   || 0.0440 |  0.5455 |  0.0176 | 0.9555 || 0.0506 |  0.6598 |  0.0186 | 0.9272 ||  27.13  | 0.525 | 00:08.72
   5   || 0.0429 |  0.5288 |  0.0174 | 0.9587 || 0.0508 |  0.6631 |  0.0186 | 0.9275 ||  25.60  | 0.457 | 00:10.98
   6   || 0.0418 |  0.5145 |  0.0170 | 0.9610 || 0.0511 |  0.6685 |  0.0186 |

### Save Model

In [6]:
model_path = base_artifacts / 'MF_Models'
model.save(path=model_path / ('MF_model_simulation.pt'), note=None)

dict_out = {
    'n_users': n_users,
    'n_items': n_items,
    'n_factors': 40,
}

with open(model_path / f'MF_params_simulation.pkl', 'wb') as f:
    pickle.dump(dict_out, f)

### Load Model

In [7]:
model_path = base_artifacts / 'MF_Models'
with open(model_path / f'MF_params_simulation.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

In [8]:
loaded_model = MF.MatrixFactorizationTorch(
    n_users=loaded_params['n_users'], 
    n_items=loaded_params['n_items'], 
    n_factors=loaded_params['n_factors']
)

model_path = base_artifacts / 'MF_Models'
loaded_model.load(path=model_path / ('MF_model_simulation.pt'))

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            5000
Number of items:            3000
Number of factors:          40
Learning rate:              0.005
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           50
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-08-15 09:33:04
